# Day 31: Store Layer: an Entries Table with Constraints, a No-op Upsert & One Transaction per Batch (24 Nov 2026)

**Week 5 (consolidation week), Phase 1 wrap-up: layer 2 of the thin-sync project.**

**Goal:** store what the fetch layer returns in Postgres the way a production sync does: a table that refuses bad rows, an upsert that writes only what changed, one transaction per batch, and a report that proves a re-run changes 0 rows.

**Deliverables (in this folder):**
- `schema.sql`: the `entries` table with its constraints (TODO 3)
- `load.py`: TODOs 1, 2 and 4 to 7, with the 22 given cases in `test_load.py` green
- `test_mine.py`: 2 tests of your own
- `dsa.py`: `plan_upsert`, `longest_unique_run`

**Before running this notebook:** start Postgres and set `DATABASE_URL` (for example `export DATABASE_URL=postgresql://localhost/fde`) in the terminal you start Jupyter from. Run the cells in order: Concept 2 and 3 build on the `products` table from Concept 1.

## Recap answers (fill in before running anything)

1. httpx gets a 404 from the CMS. Does it raise an exception? Name the two kinds of failure from Day 30 and which exception family the second kind comes from.
2. Why is `Entry.tags` a tuple and not a list, and what does `frozen=True` stop?
3. Day 30's `from_api()` rejects "2026-11-20T10:00:00" (no timezone). Why is a timestamp without a timezone a problem once it reaches a database?

*Your answers:*

1. 
2. 
3. 

## Concept 1: A table designed for synced data

Your table is a copy of someone else's system, and many things will write to it. **Constraints** protect it from every code path: Python validation is the form check in the browser, constraints are the server-side check you never skip.

Four decisions: the **key** is the CMS identity, `PRIMARY KEY (uid, locale)` for Contentstack; **NOT NULL + CHECK** refuse missing and wrong values (name the important ones, the name shows up in every error); **types that keep meaning** (`timestamptz`, `text[]`, integer paise); **schema as code** with `IF NOT EXISTS` so the file can run on every start.

In [1]:
# Write the Concept 1 SQL to a scratch file, then run it with psql.
from pathlib import Path

Path("scratch").mkdir(exist_ok=True)  # no error if it already exists

In [2]:
%%writefile scratch/c1.sql
-- Concept 1: a sync target table whose rules live in the database.
DROP TABLE IF EXISTS products;  -- start clean each time you run the file
CREATE TABLE IF NOT EXISTS products (
    sku         text        NOT NULL,
    -- DEFAULT fills the value when an INSERT leaves the column out
    locale      text        NOT NULL DEFAULT 'en-us',
    -- btrim() strips spaces at both ends, like .trim() in JS
    name        text        NOT NULL CHECK (btrim(name) <> ''),
    -- money as whole paise: integers are exact, floats are not (Day 1)
    price_paise integer     NOT NULL CHECK (price_paise >= 0),
    tags        text[]      NOT NULL DEFAULT '{}',  -- a text array
    updated_at  timestamptz NOT NULL,  -- when the CMS changed it
    synced_at   timestamptz NOT NULL DEFAULT now(),  -- when WE wrote it
    -- the natural key from the CMS: one row per product per locale
    PRIMARY KEY (sku, locale),
    -- a NAMED constraint: the name appears in every error message
    CONSTRAINT products_locale_format
        CHECK (locale ~ '^[a-z]{2}-[a-z]{2}$')
);

INSERT INTO products (sku, locale, name, price_paise, updated_at)
VALUES ('SKU-1', 'en-us', 'Kettle', 249900, '2026-11-20T09:30:00Z');
-- -> INSERT 0 1
INSERT INTO products (sku, locale, name, price_paise, updated_at)
VALUES ('SKU-1', 'hi-in', 'Kettle', 249900, '2026-11-20T15:00:00+05:30');
-- -> INSERT 0 1   same sku, other locale: a different key
INSERT INTO products (sku, locale, name, price_paise, updated_at)
VALUES ('SKU-1', 'en-us', 'Kettle v2', 1, '2026-11-21T09:30:00Z');
-- -> ERROR:  duplicate key value violates unique constraint
-- ->         "products_pkey"
INSERT INTO products (sku, locale, name, price_paise, updated_at)
VALUES ('SKU-2', 'EN_US', '  ', -5, '2026-11-21T09:30:00Z');
-- -> ERROR:  new row for relation "products" violates check
-- ->         constraint "products_locale_format"
-- GOTCHA: this row breaks three rules, but Postgres reports only one.

SET timezone = 'Asia/Kolkata';  -- show times the way Mumbai reads them
SELECT sku, locale, updated_at FROM products ORDER BY locale;
-- ->   sku  | locale |        updated_at
-- ->  SKU-1 | en-us  | 2026-11-20 15:00:00+05:30
-- ->  SKU-1 | hi-in  | 2026-11-20 15:00:00+05:30
-- REMEMBER: timestamptz stores a moment. 09:30Z and 15:00+05:30 are equal.


Writing scratch/c1.sql


In [3]:
# Run the file with psql. The ERROR lines are expected: they show the
# constraints doing their job. (! runs a shell command from a notebook.)
!psql "$DATABASE_URL" -q -f scratch/c1.sql 2>&1 | grep -v NOTICE

psql:scratch/c1.sql:28: ERROR:  duplicate key value violates unique constraint "products_pkey"
DETAIL:  Key (sku, locale)=(SKU-1, en-us) already exists.
psql:scratch/c1.sql:32: ERROR:  new row for relation "products" violates check constraint "products_locale_format"
DETAIL:  Failing row contains (SKU-2, EN_US,   , -5, {}, 2026-11-21 15:00:00+05:30, 2026-11-24 08:07:56.148268+05:30).
  sku  | locale |        updated_at         
-------+--------+---------------------------
 SKU-1 | en-us  | 2026-11-20 15:00:00+05:30
 SKU-1 | hi-in  | 2026-11-20 15:00:00+05:30
(2 rows)



**Remember:** the database is the last line of defence. Key on the CMS's identity (uid + locale), refuse bad rows with NOT NULL and CHECK, store moments as `timestamptz`, and keep the schema in a re-runnable file.

## Concept 2: The upsert that writes nothing when nothing changed

`ON CONFLICT ... DO UPDATE ... WHERE (stored) IS DISTINCT FROM (incoming)` makes a re-run a no-op: identical rows are not written and RETURNING gives nothing back for them. `IS DISTINCT FROM` is NULL-safe, `<>` is not. `RETURNING (xmax = 0) AS inserted` tells an insert from an update. A **stale guard** (`EXCLUDED.updated_at >= stored.updated_at`) stops an older version from overwriting a newer one.

In [4]:
%%writefile scratch/c2.sql
-- Concept 2: an upsert that writes only when something really changed.
-- Run after c1.sql: products holds SKU-1 in en-us and hi-in.
INSERT INTO products AS p (sku, locale, name, price_paise, tags, updated_at)
VALUES ('SKU-1', 'en-us', 'Kettle', 249900, '{}', '2026-11-20T09:30:00Z'),
       ('SKU-3', 'en-us', 'Toaster', 199900, '{kitchen}',
        '2026-11-22T08:00:00Z')
-- the conflict target must match a PRIMARY KEY or UNIQUE constraint
ON CONFLICT (sku, locale) DO UPDATE
-- EXCLUDED = the row you tried to insert; p = the row already stored
SET name        = EXCLUDED.name,
    price_paise = EXCLUDED.price_paise,
    tags        = EXCLUDED.tags,
    updated_at  = EXCLUDED.updated_at,
    synced_at   = now()
-- the WHERE turns "update" into "update only if different"
WHERE (p.name, p.price_paise, p.tags, p.updated_at)
      IS DISTINCT FROM
      (EXCLUDED.name, EXCLUDED.price_paise, EXCLUDED.tags,
       EXCLUDED.updated_at)
-- xmax = 0 is true for a row this statement inserted (see below)
RETURNING sku, locale, (xmax = 0) AS inserted;
-- ->   sku  | locale | inserted
-- ->  SKU-3 | en-us  | t
-- -> (1 row)       SKU-1 was identical: no row written, none returned

-- The same statement with a new price for SKU-1:
INSERT INTO products AS p (sku, locale, name, price_paise, tags, updated_at)
VALUES ('SKU-1', 'en-us', 'Kettle', 229900, '{}', '2026-11-23T09:30:00Z'),
       ('SKU-3', 'en-us', 'Toaster', 199900, '{kitchen}',
        '2026-11-22T08:00:00Z')
ON CONFLICT (sku, locale) DO UPDATE
SET name        = EXCLUDED.name,
    price_paise = EXCLUDED.price_paise,
    tags        = EXCLUDED.tags,
    updated_at  = EXCLUDED.updated_at,
    synced_at   = now()
WHERE (p.name, p.price_paise, p.tags, p.updated_at)
      IS DISTINCT FROM
      (EXCLUDED.name, EXCLUDED.price_paise, EXCLUDED.tags,
       EXCLUDED.updated_at)
RETURNING sku, locale, (xmax = 0) AS inserted;
-- ->   sku  | locale | inserted
-- ->  SKU-1 | en-us  | f        an UPDATE: the row existed
-- -> (1 row)       SKU-3 unchanged this time, so it is not returned

-- Why IS DISTINCT FROM and not <> ? Because of NULL:
SELECT NULL <> NULL AS not_equal, NULL IS DISTINCT FROM NULL AS distinct_;
-- ->  not_equal | distinct_
-- ->            | f          NULL <> NULL is NULL (unknown), not true


Writing scratch/c2.sql


In [5]:
# Run Concept 2 against the products table from Concept 1.
!psql "$DATABASE_URL" -q -f scratch/c2.sql

  sku  | locale | inserted 
-------+--------+----------
 SKU-3 | en-us  | t
(1 row)

  sku  | locale | inserted 
-------+--------+----------
 SKU-1 | en-us  | f
(1 row)

 not_equal | distinct_ 
-----------+-----------
           | f
(1 row)



| Statement | Second run, same data | Use it for |
|---|---|---|
| Plain INSERT | Fails on the first duplicate key | One-time imports |
| ON CONFLICT DO NOTHING | Writes nothing, never applies changes | Append-only logs |
| DO UPDATE without WHERE | Rewrites every row | Rarely |
| DO UPDATE ... WHERE IS DISTINCT FROM | Writes nothing, reports 0 changed | Syncing a copy (today) |

**Remember:** idempotent means the second run writes nothing and says so. Compare with IS DISTINCT FROM, and never let an older version win.

## Concept 3: psycopg 3, Python types and one transaction per batch

psycopg adapts Python values to Postgres types: aware datetime to timestamptz, **list** to array (a tuple is not an array), None to NULL. With `autocommit=True`, the only multi-statement transactions are your `with conn.transaction():` blocks: they commit on success and roll back if an exception escapes.

In [6]:
# Concept 3: psycopg 3 sends Python values; Postgres gets real types.
import os
from datetime import datetime

import psycopg  # pip install "psycopg[binary]"  (the v3 driver)

# FDE TIP: the connection string comes from the environment, never the code
url = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")

# "with" closes the connection at the end, like a finally block in JS
with psycopg.connect(url) as conn:
    # make the output the same on every laptop: show times in UTC
    conn.execute("SET TIME ZONE 'UTC'")
    ts = datetime.fromisoformat("2026-11-20T15:00:00+05:30")
    row = conn.execute(
        # %s placeholders: the values travel separately from the SQL text,
        # so no quoting by hand and no SQL injection (Day 18)
        "SELECT %s::timestamptz, %s::text[]",
        [ts, ["sale", "winter"]],  # a Python list becomes a Postgres array
    ).fetchone()
    print(row[0])  # -> 2026-11-20 09:30:00+00:00  (same moment, UTC view)
    print(row[1])  # -> ['sale', 'winter']  (an array comes back as a list)

    # GOTCHA: a tuple is NOT sent as an array. Convert with list() first.
    try:
        conn.execute("SELECT %s::text[]", [("sale", "winter")])
    except psycopg.errors.InvalidTextRepresentation as exc:
        print(type(exc).__name__)  # -> InvalidTextRepresentation
        conn.rollback()  # the failed statement aborted the transaction

2026-11-20 09:30:00+00:00
['sale', 'winter']
InvalidTextRepresentation


In [7]:
# Concept 3: a batch upsert inside ONE transaction, counted with RETURNING.
import os

import psycopg
from psycopg.rows import dict_row  # rows as dicts, like JSON objects in JS

url = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")

UPSERT = """
INSERT INTO products AS p (sku, locale, name, price_paise, updated_at)
VALUES (%(sku)s, %(locale)s, %(name)s, %(price)s, %(updated_at)s)
ON CONFLICT (sku, locale) DO UPDATE
SET name = EXCLUDED.name, price_paise = EXCLUDED.price_paise,
    updated_at = EXCLUDED.updated_at, synced_at = now()
WHERE (p.name, p.price_paise) IS DISTINCT FROM
      (EXCLUDED.name, EXCLUDED.price_paise)
RETURNING (xmax = 0) AS inserted
"""

rows = [  # named placeholders %(sku)s read these dict keys
    {"sku": "SKU-1", "locale": "en-us", "name": "Kettle",
     "price": 229900, "updated_at": "2026-11-23T09:30:00Z"},
    {"sku": "SKU-4", "locale": "en-us", "name": "Mixer",
     "price": 549900, "updated_at": "2026-11-23T10:00:00Z"},
]

with psycopg.connect(url, row_factory=dict_row) as conn:
    # REMEMBER: everything inside this block commits together or not at all
    with conn.transaction():
        with conn.cursor() as cur:
            inserted = updated = 0
            for params in rows:
                cur.execute(UPSERT, params)
                result = cur.fetchone()  # None when the WHERE skipped it
                if result is None:
                    continue  # unchanged: no row was written
                if result["inserted"]:
                    inserted += 1
                else:
                    updated += 1
    print(inserted, updated, len(rows) - inserted - updated)
    # -> 1 0 1   first run: SKU-4 is new, SKU-1 already had these values

1 0 1


In [8]:
# Concept 3: one bad row inside a transaction rolls back the whole batch.
import os

import psycopg

url = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")
SQL = ("INSERT INTO products (sku, name, price_paise, updated_at)"
       " VALUES (%s, %s, %s, now())")

with psycopg.connect(url) as conn:
    before = conn.execute("SELECT count(*) FROM products").fetchone()[0]
    conn.commit()  # end the transaction that SELECT opened
    try:
        with conn.transaction():
            conn.execute(SQL, ["SKU-5", "Blender", 399900])  # fine
            conn.execute(SQL, ["SKU-6", "Juicer", -1])  # breaks a CHECK
    except psycopg.errors.CheckViolation as exc:
        # the error names the constraint: log it, it tells you which rule
        print(exc.diag.constraint_name)  # -> products_price_paise_check
    after = conn.execute("SELECT count(*) FROM products").fetchone()[0]
    print(before, after)  # -> 4 4   SKU-5 was rolled back with SKU-6

products_price_paise_check
4 4


Run the batch cell (the second one) again and it prints `0 0 2`: nothing was written the second time.

**Remember:** one batch, one transaction: all rows or none. Convert tuples with `list()`, keep datetimes as datetimes, and after an error in an open transaction, roll back before anything else.

## Concept 4: Idempotency as a test you can show

Each test gets a fresh Postgres **schema** (a namespace) selected with `SET search_path`, so tests never share rows and never touch real tables; with no database they **skip** instead of failing. The given `test_load.py` uses the same fixture.

In [9]:
%%writefile scratch/test_idempotent.py
# Concept 4: test "run twice, change nothing" against a real Postgres.
import os
import uuid

import psycopg
import pytest

URL = os.environ.get("DATABASE_URL", "postgresql://localhost/fde")
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS products (
    sku  text PRIMARY KEY,
    name text NOT NULL CHECK (btrim(name) <> '')
)"""
UPSERT = """
INSERT INTO products AS p (sku, name) VALUES (%s, %s)
ON CONFLICT (sku) DO UPDATE SET name = EXCLUDED.name
WHERE p.name IS DISTINCT FROM EXCLUDED.name
RETURNING (xmax = 0) AS inserted"""


def load(conn: psycopg.Connection, rows: list[tuple[str, str]]) -> dict:
    """Upsert rows in one transaction; count what really happened."""
    counts = {"inserted": 0, "updated": 0, "unchanged": 0}
    with conn.transaction():
        for sku, name in rows:
            result = conn.execute(UPSERT, [sku, name]).fetchone()
            if result is None:
                counts["unchanged"] += 1
            elif result[0]:
                counts["inserted"] += 1
            else:
                counts["updated"] += 1
    return counts


@pytest.fixture
def conn():
    # A fresh, uniquely named schema per test: tests never see each
    # other's rows, and they never touch your real tables.
    schema = f"test_{uuid.uuid4().hex[:8]}"
    try:
        c = psycopg.connect(URL, autocommit=True)
    except psycopg.OperationalError:
        pytest.skip("no Postgres at DATABASE_URL")  # skip, do not fail
    c.execute(f'CREATE SCHEMA "{schema}"')
    # search_path: where unqualified table names are looked up,
    # like the module resolution order for imports in Node
    c.execute(f'SET search_path TO "{schema}"')
    c.execute(SCHEMA_SQL)
    yield c  # the test runs here (Day 10 fixtures)
    c.execute(f'DROP SCHEMA "{schema}" CASCADE')  # clean up afterwards
    c.close()


ROWS = [("SKU-1", "Kettle"), ("SKU-2", "Toaster")]


def test_first_load_inserts_everything(conn):
    assert load(conn, ROWS) == {"inserted": 2, "updated": 0, "unchanged": 0}


def test_second_load_changes_nothing(conn):
    load(conn, ROWS)
    # REMEMBER: idempotent = the second run reports zero writes
    assert load(conn, ROWS) == {"inserted": 0, "updated": 0, "unchanged": 2}


def test_a_changed_row_is_one_update(conn):
    load(conn, ROWS)
    changed = [("SKU-1", "Kettle 2L"), ("SKU-2", "Toaster")]
    assert load(conn, changed) == {"inserted": 0, "updated": 1,
                                   "unchanged": 1}


def test_bad_row_rolls_back_the_batch(conn):
    with pytest.raises(psycopg.errors.CheckViolation):
        load(conn, [("SKU-1", "Kettle"), ("SKU-2", "   ")])
    count = conn.execute("SELECT count(*) FROM products").fetchone()[0]
    assert count == 0  # SKU-1 did not survive on its own


Writing scratch/test_idempotent.py


In [10]:
# Run pytest on the scratch file, as you would in a terminal.
import subprocess
import sys

# sys.executable = this notebook's Python, so the right packages are used
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no",
     "scratch/test_idempotent.py"],
    capture_output=True, text=True)
print(result.stdout.strip().splitlines()[-1])  # the summary line

4 passed in 0.09s


**Remember:** a re-run claim is only as good as the test that proves it. Assert the second report is all zeros and that `synced_at` did not move.

## Check your understanding (do NOT run until you have predicted)

The five questions are in the Day 31 session in the master doc (`check.py`). Write your predictions here first, then save the file as `check.py` and run it.

*Predictions:*

1: 
2: 
3: 
4: 
5: 

## Today's task

Open `schema.sql`, `load.py`, `test_mine.py` and `dsa.py` in this folder. Each has the brief, acceptance criteria and step-by-step hints. Run `python -m pytest -q` after every TODO.

## Notes and questions

*(Write what surprised you, and anything to ask in the Day 31 review.)*